# Image Similarity Analysis using CLIP

This notebook analyzes images in the `images/` folder and computes cosine similarity between them using OpenAI's CLIP model.

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

## Load CLIP Model

In [ ]:
# Load CLIP model and processor
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

## Load Images from Directory

In [ ]:
# Define image folder path
image_folder = Path("images")

# Supported image extensions
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}

# Load all images from the folder
image_paths = []
for ext in image_extensions:
    image_paths.extend(image_folder.glob(f"*{ext}"))
    image_paths.extend(image_folder.glob(f"*{ext.upper()}"))

image_paths = sorted(image_paths)
print(f"Found {len(image_paths)} images")

if len(image_paths) == 0:
    print(f"No images found in {image_folder}. Please add some images to continue.")
else:
    for img_path in image_paths:
        print(f"  - {img_path.name}")

## Extract Image Embeddings

In [ ]:
def get_image_embedding(image_path):
    """
    Extract CLIP embedding for a single image.
    """
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
    
    # Normalize the embedding
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    return image_features.cpu().numpy()

# Extract embeddings for all images
embeddings = []
image_names = []

if len(image_paths) > 0:
    print("Extracting embeddings...")
    for img_path in image_paths:
        print(f"Processing: {img_path.name}")
        embedding = get_image_embedding(img_path)
        embeddings.append(embedding)
        image_names.append(img_path.name)
    
    embeddings = np.vstack(embeddings)
    print(f"\nExtracted embeddings shape: {embeddings.shape}")

## Compute Cosine Similarity Matrix

In [ ]:
if len(image_paths) > 0:
    # Compute cosine similarity matrix
    similarity_matrix = cosine_similarity(embeddings)
    
    print("Cosine Similarity Matrix:")
    print(similarity_matrix)
    print(f"\nMatrix shape: {similarity_matrix.shape}")

## Visualize Similarity Matrix

In [ ]:
if len(image_paths) > 0:
    # Create a heatmap of the similarity matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        similarity_matrix,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        xticklabels=image_names,
        yticklabels=image_names,
        vmin=0,
        vmax=1,
        square=True,
        cbar_kws={"label": "Cosine Similarity"}
    )
    plt.title("Image Similarity Matrix (CLIP)", fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Find Most Similar Image Pairs

In [ ]:
if len(image_paths) > 1:
    # Get upper triangle indices (excluding diagonal)
    upper_tri_indices = np.triu_indices_from(similarity_matrix, k=1)
    
    # Create list of (similarity, image1, image2) tuples
    similarities = []
    for i, j in zip(*upper_tri_indices):
        similarities.append((
            similarity_matrix[i, j],
            image_names[i],
            image_names[j]
        ))
    
    # Sort by similarity (descending)
    similarities.sort(reverse=True)
    
    # Display top 10 most similar pairs
    print("Top 10 Most Similar Image Pairs:")
    print("=" * 70)
    for idx, (sim, img1, img2) in enumerate(similarities[:10], 1):
        print(f"{idx}. {img1} <-> {img2}")
        print(f"   Similarity: {sim:.4f}")
        print()
    
    # Display least similar pairs
    print("\nTop 10 Least Similar Image Pairs:")
    print("=" * 70)
    for idx, (sim, img1, img2) in enumerate(similarities[-10:], 1):
        print(f"{idx}. {img1} <-> {img2}")
        print(f"   Similarity: {sim:.4f}")
        print()

## Display Image Grid

In [ ]:
if len(image_paths) > 0:
    # Display all images in a grid
    n_images = len(image_paths)
    cols = min(4, n_images)
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    if n_images == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if rows > 1 else axes
    
    for idx, img_path in enumerate(image_paths):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name, fontsize=10)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()